# Capítulo 29 · Laboratório 29: gestão quantitativa do risco, o tamanho da mão
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (40 ações desde 2005, Ibovespa e a Americanas sem ajuste). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap29.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. A limpeza é a mesma do Capítulo 28. As operações são rompimentos de 20 dias com stop de 3 ATR, medidas em múltiplos de R (o risco inicial), com custo de 0,10%. Os sorteios usam semente fixa: os números se repetem a cada execução.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import brentq, minimize_scalar

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
FIM, CUSTO, PRAZO = "2026-10-01", 0.001, 20
CACHE = "dados_cap29.pkl"
rng = np.random.default_rng(31)            # sorteios das Partes 3 e 4 (semente fixa)


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    dados, ibov, amer = pd.read_pickle(CACHE)
else:
    dados = baixa(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, group_by="ticker")
    ibov = baixa("^BVSP", start="2005-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    amer = baixa("AMER3.SA", start="2022-12-01", end="2023-02-01", auto_adjust=False, multi_level_index=False)
    pd.to_pickle((dados, ibov, amer), CACHE)


CORRIGIDAS = {}                            # o que a limpeza mudou (registro da Parte 7; sem repetir)
SUSPEITOS = {"EQTL3.SA": ("2010-08-25", "queda de 40% sem volta (provável ajuste de bonificação ou desdobramento faltando)")}


def limpa(d, nome=""):
    """Limpeza do Capítulo 26, mais três regras para dado sujo que engana o teste de stops:
    1) máxima impossível: pavio de mais de 50% acima da abertura e do fechamento vira o maior dos dois;
    2) desdobramento ou grupamento sem ajuste: salto de mais de 2 vezes (ou menos da metade) da abertura em relação ao
       fechamento anterior, sem volta nos 5 pregões seguintes, corrige o histórico anterior pela mesma proporção;
    3) buraco nos dados: com mais de 15 dias corridos sem cotação, só fica o trecho contínuo depois do último buraco."""
    d = d.dropna().copy()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    d = d[ok & (d["Volume"] > 0)] if "Volume" in d else d[ok]
    px = ["Open", "High", "Low", "Close"]
    topo = d[["Open", "Close"]].max(axis=1)
    alta = d["High"] > 1.5 * topo
    for dt in d.index[alta]:
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: máxima {d.at[dt, 'High']:.2f} com fechamento {d.at[dt, 'Close']:.2f}", None)
    d.loc[alta, "High"] = topo[alta]
    salto = d["Open"] / d["Close"].shift()
    dura = d["Close"].shift(-5) / d["Close"].shift()
    for dt in d.index[((salto < 0.5) | (salto > 2)) & ((dura < 0.6) | (dura > 1.6))]:
        f_ = salto.loc[dt]
        d.loc[d.index < dt, px] *= f_
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: salto de {f_:.4f} vezes sem volta (ajuste que faltou); histórico"
                          " anterior corrigido", None)
    if nome == "SUZB3.SA":                 # antes disso, preço parado no Yahoo (Capítulo 26)
        d = d.loc["2017-11-10":]
    if nome in SUSPEITOS:                  # queda sem volta que parece ajuste faltando, mas sem prova: fica fora
        dt, motivo = SUSPEITOS[nome]
        CORRIGIDAS.setdefault(f"{nome} {pd.Timestamp(dt):%d/%m/%Y}: {motivo}; usa só a partir dessa data", None)
        d = d.loc[dt:]
    buraco = d.index.to_series().diff().dt.days
    if (buraco > 15).any():
        ini = buraco[buraco > 15].index[-1]
        CORRIGIDAS.setdefault(f"{nome}: {int((buraco > 15).sum())} buracos de mais de 15 dias; usa só a partir de {ini:%d/%m/%Y}", None)
        d = d.loc[ini:]
    return d



mostra("Parte 1. Dados até 30/09/2026; limpeza do Capítulo 28 (o que ela corrige aparece na Parte 7)")


def atr(h, l, c, n=20):
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()

## Parte 2: as operações em múltiplos de R

In [ ]:
ops = []
for t in TICKERS:
    d = limpa(dados[t], t)
    O, H, L, C = (d[x].to_numpy() for x in ("Open", "High", "Low", "Close"))
    A = atr(d["High"], d["Low"], d["Close"]).to_numpy()
    c = d["Close"]
    sinal = ((c > d["High"].rolling(20).max().shift(1)) & (c > c.rolling(200).mean())).to_numpy()
    livre = 0
    for i in np.flatnonzero(sinal):
        if i < 210 or i > len(C) - PRAZO - 2 or i < livre:
            continue
        e, risco = C[i], 3 * A[i]
        stop, alvo = e - risco, e + risco
        # (a) posição inteira: stop de 3 ATR ou saída em 20 pregões
        res_a = None
        for k in range(i + 1, i + PRAZO + 1):
            if L[k] <= stop:
                res_a = min(stop, O[k]); break
        if res_a is None:
            res_a = C[i + PRAZO]
        # (b) realiza metade no alvo de 1R e leva o stop da outra metade para a entrada
        metade, st, res_b = None, stop, None
        for k in range(i + 1, i + PRAZO + 1):
            if L[k] <= st:
                saida = min(st, O[k])
                res_b = saida if metade is None else (metade + saida) / 2
                break
            if metade is None and H[k] >= alvo:
                metade, st = max(alvo, O[k]), e
        if res_b is None:
            res_b = C[i + PRAZO] if metade is None else (metade + C[i + PRAZO]) / 2
        custo_r = CUSTO * e / risco
        ops.append({"ativo": t, "data": d.index[i], "R_inteira": (res_a - e) / risco - custo_r,
                    "R_metade": (res_b - e) / risco - custo_r, "risco_pct": risco / e})
        livre = i + PRAZO
P = pd.DataFrame(ops)
R = P["R_inteira"].to_numpy()


def resumo(x):
    g, p = x[x > 0], x[x <= 0]
    W, V = len(g) / len(x), g.mean() / -p.mean()
    return W, g.mean(), p.mean(), V, x.mean(), W - (1 - W) / V


W, gm, pm, V, E, K = resumo(R)
mostra(f"Parte 2. {len(R):_} rompimentos de 20 dias com stop de 3 ATR (distância mediana do stop: {P['risco_pct'].median():.1%} do preço)")
mostra(f"   acerto {W:.0%} | ganho médio {gm:+.2f} R | perda média {pm:+.2f} R | payoff {V:.2f} | expectativa {E:+.3f} R"
      f" | Kelly {K:.1%} do capital arriscado por operação")
jp = P["R_inteira"].idxmin()
mostra(f"   pior operação {R.min():+.2f} R ({P.loc[jp, 'ativo'].replace('.SA', '')}, {P.loc[jp, 'data']:%d/%m/%Y}; gap além do stop) | operações piores que -1,5 R: {np.mean(R < -1.5):.1%}")

bb = limpa(dados["BBAS3.SA"], "BBAS3.SA")
pb, ab = bb["Close"].iloc[-1], atr(bb["High"], bb["Low"], bb["Close"]).iloc[-1]
mostra(f"   lote no exemplo de BBAS3 ({bb.index[-1]:%d/%m/%Y}): preço R$ {pb:.2f}; ATR(20) R$ {ab:.2f}; perda máxima R$ 200")
for rot, st in (("3 ATR", pb - 3 * ab), ("mínima dos 10 pregões", bb["Low"].iloc[-10:].min())):
    dist = pb - st
    lote = int(200 // dist)
    mostra(f"      stop por {rot}: R$ {st:.2f} (distância R$ {dist:.2f}, {dist / pb:.1%}) -> {lote} ações, R$ {lote * pb:_.0f}")

## Parte 3: a incerteza do Kelly

In [ ]:
ks = []
for _ in range(2000):
    s = rng.choice(R, size=len(R))
    ks.append(resumo(s)[5])
ks = np.array(ks)
mostra(f"\nParte 3. Kelly reamostrado (2_000 vezes): mediana {np.median(ks):.1%}, 90% entre {np.percentile(ks, 5):.1%} e"
      f" {np.percentile(ks, 95):.1%}; Kelly negativo (não operar) em {np.mean(ks <= 0):.0%} das amostras")
k200 = [resumo(rng.choice(R, size=200))[5] for _ in range(2000)]
mostra(f"   com só 200 operações: 90% entre {np.percentile(k200, 5):.1%} e {np.percentile(k200, 95):.1%}")

## Parte 4: quanto arriscar por operação

In [ ]:
N, CAM = 250, 5000
mostra(f"\nParte 4. {CAM:_} caminhos de {N} operações sorteadas das reais; risco fixo por operação (fração do capital no stop)")
mostra("   risco      | capital final mediano | 5% piores | pior queda mediana | chance de cair 50% | chance de perder 80%")
cresc = lambda f, x: np.mean(np.log1p(f * x))              # crescimento médio do log do capital por operação
f_exato = minimize_scalar(lambda f: -cresc(f, R), bounds=(1e-4, 1 / -R.min() - 1e-6), method="bounded").x
fracs = [("0,5%", 0.005), ("1%", 0.01), ("2%", 0.02), ("5%", 0.05), (f"meio Kelly ({f_exato / 2:.1%})", f_exato / 2),
         ("10%", 0.10), (f"Kelly ({f_exato:.1%})", f_exato)]
for rot, f in fracs:
    amostra = rng.choice(R, size=(CAM, N))
    mult = np.clip(1 + f * amostra, 1e-9, None)
    eq = np.cumprod(mult, axis=1)
    dd = (eq / np.maximum.accumulate(eq, axis=1) - 1).min(axis=1)
    mostra(f"   {rot:<10} | {np.median(eq[:, -1]):>20.2f}x | {np.percentile(eq[:, -1], 5):>8.2f}x | {np.median(dd):>17.0%} |"
          f" {np.mean(dd <= -0.5):>17.0%} | {np.mean(eq.min(axis=1) <= 0.2):>18.0%}")

In [ ]:
import matplotlib.pyplot as plt
f = np.linspace(0.001, 0.99 / -R.min(), 300)
plt.figure(figsize=(8, 4))
plt.plot(f * 100, [np.mean(np.log1p(x * R)) * 100 for x in f], color="black")
plt.axhline(0, color="gray", lw=0.6)
plt.axvline(f_exato * 100, ls="--", color="gray")
plt.xlabel("capital arriscado por operação (%)")
plt.ylabel("crescimento médio por operação (%)")
plt.show()

## Parte 5: realizar metade ou segurar

In [ ]:
mostra("\nParte 5. Realizar metade no alvo de 1R (stop da outra metade na entrada) x segurar a posição inteira")
for rot, col in (("posição inteira", "R_inteira"), ("metade no alvo de 1R", "R_metade")):
    x = P[col].to_numpy()
    w_, g_, p_, v_, e_, k_ = resumo(x)
    mostra(f"   {rot:<22} acerto {w_:.0%} | expectativa {e_:+.3f} R | desvio {x.std():.2f} R | 5% piores {np.percentile(x, 5):+.2f} R")

## Parte 6: correlação na crise

In [ ]:
Rb = pd.DataFrame({t: limpa(dados[t + ".SA"], t + ".SA")["Close"].pct_change() for t in B3}).loc["2006":]
ib = ibov["Close"].pct_change().reindex(Rb.index)


def corr_media(df):
    m = df.dropna(axis=1, thresh=int(len(df) * 0.8)).corr().to_numpy()
    return m[np.triu_indices_from(m, 1)].mean()


movel = pd.Series([corr_media(Rb.iloc[k - 60:k]) for k in range(60, len(Rb), 5)], index=Rb.index[60::5])
mostra(f"\nParte 6. Correlação média entre as 20 ações da B3 (retornos diários, janelas de 60 pregões, 2006 a 2026)")
mostra(f"   mediana das janelas: {movel.median():.2f} | 10% mais baixas até {movel.quantile(0.1):.2f} | 10% mais altas acima de {movel.quantile(0.9):.2f}")
c08, c20 = corr_media(Rb.loc["2008-10-01":"2008-10-31"]), corr_media(Rb.loc["2020-03-01":"2020-03-31"])
mostra(f"   out/2008: {c08:.2f} | mar/2020: {c20:.2f}")
for rot, rho in (("correlação zero", 0.0), (f"correlação típica ({movel.median():.2f})", movel.median()), (f"correlação de mar/2020 ({c20:.2f})", c20), ("correlação 1", 1.0)):
    mostra(f"   10 posições com 1% de risco cada, {rot}: risco conjunto equivalente a {np.sqrt(10 + 90 * rho):.1f}% do capital")

## Parte 7: o gap e o tamanho da posição

In [ ]:
a = amer["Close"]
mostra(f"\nParte 7. Americanas (AMER3), 12/01/2023: fechamento anterior {a.loc['2023-01-11']:.0f}, abertura"
      f" {amer.loc['2023-01-12', 'Open']:.0f} ({amer.loc['2023-01-12', 'Open'] / a.loc['2023-01-11'] - 1:+.0%}),"
      f" fechamento {a.loc['2023-01-12']:.0f} ({a.loc['2023-01-12'] / a.loc['2023-01-11'] - 1:+.0%}) (preços sem ajuste de proventos)")
mostra("   perda em % do capital, com risco planejado de 1% e stop a X% da entrada, se o papel abrir Y% abaixo:")
mostra("   stop    | posição (% do capital) | gap de -20% | gap de -50% | gap de -77%")
for s in (0.02, 0.05, 0.10, 0.20):
    pos = 0.01 / s
    mostra(f"   {s:>5.0%}  | {pos:>21.0%} | {pos * 0.20:>10.1%} | {pos * 0.50:>10.1%} | {pos * 0.77:>10.1%}")
lin = []
for t in TICKERS:
    d = limpa(dados[t], t)
    r = d["Close"].pct_change()
    lacuna = d.index.to_series().diff().dt.days
    i = r.idxmin()
    lin.append((t.replace(".SA", ""), i, r.min(), int(lacuna.loc[i])))
W6 = pd.DataFrame(lin, columns=["ativo", "data", "queda", "dias_desde_o_pregao_anterior"]).sort_values("queda")
mostra("   pior dia de cada ação desde 2005, depois da limpeza (as oito piores):")
for _, x in W6.head(8).iterrows():
    mostra(f"      {x['ativo']:<6} {x['data']:%d/%m/%Y} {x['queda']:+.0%} (dias corridos desde o pregão anterior: {x['dias_desde_o_pregao_anterior']})")
mostra(f"   pior dia mediano {W6['queda'].median():+.0%}; ações com algum dia de -20% ou pior:"
       f" {(W6['queda'] <= -0.20).sum()} de {len(W6)}")
mostra(f"   o que a limpeza corrigiu ({len(CORRIGIDAS)} casos; sem ela, as piores quedas eram erros do provedor):")
for linha in CORRIGIDAS:
    mostra("      " + linha)

## Parte 8: as contas da 3ª edição

In [ ]:
G, Vx = 0.70, 450 / 150
mostra(f"\nParte 8. Kelly do exemplo da 3ª ed. (G = 70%, ganho médio R$ 450, perda média R$ 150): {G - (1 - G) / Vx:.0%} (como no livro; a 3ª edição dizia 50%)")
mostra(f"   Roleta americana (38 casas, paga 35 por 1): {(1 * 35 - 37 * 1) / 38:+.2%} por aposta | europeia (37 casas):"
      f" {(1 * 35 - 36 * 1) / 37:+.2%} (como no livro; a 3ª edição dizia -2,6% para a americana)")

## Parte 9: Kelly com vantagem estimada (Crane, 2026)

In [ ]:
# Ideia: escolher a fração de Kelly pelo stop da estratégia (PERDA_MAX) e pela chance aceitável
# de abandonar um método que funciona (PI). Depois, encolher pela dúvida sobre a vantagem (kappa_min).

cresc = lambda f, x: np.mean(np.log1p(f * x))              # crescimento médio do log do capital por operação
teto = 1 / -R.min()                                         # acima disso a pior operação zera a conta
f_exato = minimize_scalar(lambda f: -cresc(f, R), bounds=(1e-4, teto - 1e-6), method="bounded").x


def raiz_cramer(f, x):
    """theta > 0 tal que a média de (1 + f*R)^(-theta) seja 1 (Crane, eq. 10 e Remark 39)."""
    psi = lambda t: np.mean((1 + f * x) ** (-t)) - 1
    hi = 1.0
    while psi(hi) < 0:
        hi *= 2
    return brentq(psi, 1e-6, hi)


def f_crane(perda_max, pi, x):
    """Maior fração do capital arriscada por operação com chance <= pi de bater o stop (Teorema 18)."""
    theta_min = np.log(pi) / np.log(1 - perda_max)
    f_max = teto * 0.999                                    # acima do ponto de crescimento zero não há raiz positiva
    if cresc(f_max, x) < 0:
        f_max = brentq(lambda f: cresc(f, x), f_exato, f_max) * 0.999
    return brentq(lambda f: raiz_cramer(f, x) - theta_min, 1e-5, f_max), theta_min


mostra(f"\nParte 9. Kelly com vantagem estimada (Crane, 2026)")
mostra(f"   Kelly pela fórmula simples {K:.1%} | Kelly exato (máximo do crescimento) {f_exato:.1%}"
      f" | raiz de Cramér no Kelly exato {raiz_cramer(f_exato, R):.2f} (teoria: 1)")
mostra("   stop da estratégia | chance aceitável | fração de Kelly (fórmula) | fração de Kelly (exata) | risco por operação")
for perda_max, pi in ((0.50, 0.02), (0.50, 0.05), (0.25, 0.05), (0.20, 0.05)):
    fb, tm = f_crane(perda_max, pi, R)
    a_formula = 2 * np.log(1 - perda_max) / (np.log(1 - perda_max) + np.log(pi))
    mostra(f"   {perda_max:>17.0%} | {pi:>15.0%} | {a_formula:>24.1%} | {fb / f_exato:>22.1%} | {fb:>17.2%}")

# dúvida sobre a vantagem: quanto da expectativa sobra no pior caso razoável (5% da reamostragem)
rng8 = np.random.default_rng(318)
f_base, _ = f_crane(0.50, 0.02, R)
for n in (len(R), 200):
    medias = np.array([rng8.choice(R, size=n).mean() for _ in range(4000)])
    kappa = max(np.percentile(medias, 5) / R.mean(), 0)
    mostra(f"   com {n:_} operações: kappa_min = {kappa:.2f} -> risco robusto {kappa * f_base:.2%} por operação (stop 50%, chance 2%)")


def chance_stop(f, x, perda_max=0.50, n_op=3000, cam=2000):
    eq, bateu = np.ones(cam), np.zeros(cam, bool)
    for _ in range(n_op):
        eq *= 1 + f * rng8.choice(x, size=cam)
        bateu |= eq <= 1 - perda_max
    return bateu.mean()


mostra(f"   Monte Carlo (2_000 caminhos de 3_000 operações): chance de perder 50% do capital INICIAL")
mostra("   regra                     | risco  | vantagem real | 75% dela | metade dela | sem vantagem")
tab9 = {}
kappa_tot = max(np.percentile([rng8.choice(R, size=len(R)).mean() for _ in range(4000)], 5) / R.mean(), 0)
for rot, f in (("Kelly (fórmula simples)", K), ("Kelly exato", f_exato), ("meio Kelly exato", f_exato / 2),
               ("Crane (stop 50%, 2%)", f_base), ("Crane x kappa_min", kappa_tot * f_base), ("1% fixo", 0.01)):
    linha = [chance_stop(f, R - (1 - k) * R.mean()) for k in (1.0, 0.75, 0.5, 0.0)]
    tab9[rot] = (f, linha)
    mostra(f"   {rot:<26}| {f:6.2%} | " + " | ".join(f"{c:>10.1%}" for c in linha))

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `resumo` da Parte 2 e o cálculo do lote de BBAS3 no fim da mesma célula. Mostre por que o Kelly da fórmula simples é `W - (1 - W) / V` e por que o lote é `int(200 // dist)`: a perda máxima dividida pela distância até o stop, arredondada para baixo." Confira com a Parte 2, que imprime a expectativa de +0,145 R, o Kelly de 15,8% e os lotes de 93 e 105 ações.
2. **Adaptar:** "No exemplo do lote, no fim da Parte 2, troque `BBAS3.SA` por [ATIVO]. Mantenha a perda máxima de R$ 200 e os dois stops (3 ATR e mínima dos 10 pregões). Se [ATIVO] não estiver entre as 40 ações, baixe com a função `baixa` da célula de abertura e passe por `limpa`. Não mude a fórmula do ATR." Antes de trocar, confira se o caso do livro continua igual: BBAS3 a R$ 23,07, ATR(20) de R$ 0,71 e lotes de 93 e 105 ações.

Mais pedidos deste capítulo (entender e aplicar ao seu caso) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque o stop de 3 ATR por 2 ATR. O Kelly sobe ou desce? E a pior operação em R?
2. Na Parte 4, acrescente a fração de 3% e veja onde ela cai entre 2% e 5% na chance de cair 50%.
3. Na Parte 6, troque a janela de 60 por 20 pregões. O pico de março de 2020 fica maior?
4. Na Parte 9, troque a chance aceitável de 2% por 10% e compare o risco por operação com o meio Kelly.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.